<a href="https://colab.research.google.com/github/Digitalbuilder2026/Intro-to-Computer-Vision/blob/main/Document_Cleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Cleaner · sheet music

A self-contained **Jupyter + Gradio + OpenCV** app based on your course notebook. Upload PDFs or PNG/JPEG images, adjust cleaning controls, compare the original with the result, and export PNG, JPEG, or PDF. No AI API key or GPU is needed.

Run cells in order in a **fresh Python 3.11+ kernel** and a writable folder. The cells write the shared `app.py` used for Hugging Face Spaces. Use a dedicated folder: write-file cells overwrite `app.py`, `requirements.txt`, and `README.md`, so back up any edits before rerunning them. Close the running Gradio interface before rerunning the final launch cell.

**Musical accuracy matters:** thresholding and filters can erase small dots or accidentals. Compare details with the original. This app does not invent missing notation or recognize pitches. Never replace your original PDF with a processed copy.

## 1. Install reproducible dependencies

Gradio builds the UI; OpenCV and NumPy process pixel arrays; PyMuPDF renders PDF pages; Pillow encodes images. Only `opencv-python-headless` is needed—do not deliberately install both OpenCV variants in the same environment. If incompatible libraries are already imported, restart the kernel after installation. Jupyter itself is not a Space runtime dependency.

In [1]:
%%writefile requirements.txt
gradio==6.30.0
numpy==2.5.4
opencv-python-headless==5.0.0.93
Pillow==12.3.0
PyMuPDF==1.28.2


Writing requirements.txt


In [2]:
import os
import subprocess
import sys

# The test flag is only for automated verification in an already-installed environment.
if os.environ.get('DOCUMENT_CLEANER_TEST') != '1':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
print('Using Python:', sys.executable)


Using Python: /usr/bin/python3


## 2. Upload validation and PDF-to-pixel conversion

PyMuPDF's `get_pixmap(dpi=...)` renders a page, then NumPy exposes its grayscale pixels. The app reads one page at a time, rejects locked/oversized documents, keeps original uploads unchanged, and retains physical PDF page dimensions. PNG/JPEG uploads remain at native pixel resolution. PDF processing is serialized because PyMuPDF does not support concurrent threaded use.

In [3]:
%%writefile app.py
"""Document Cleaner: a deterministic, notebook-friendly Gradio application.

The exported score never includes diagnostics or staff-line removal.
No AI inference, model downloads, credentials, or GPU are required.
"""
from __future__ import annotations

import io
import math
import os
import shutil
import tempfile
import threading
import uuid
import zipfile
from pathlib import Path

import cv2
import gradio as gr
import numpy as np
import pymupdf
from PIL import Image, ImageOps

MAX_UPLOAD_BYTES = 50 * 1024 * 1024
MAX_PAGES = 100
MAX_PAGE_PIXELS = 18_000_000
MAX_EXPORT_PIXELS = 240_000_000
PDF_LOCK = threading.RLock()  # PyMuPDF is not safe for concurrent threaded use.
cv2.setNumThreads(2)
EXPORT_PROGRESS = gr.Progress()
METHODS = ["Adaptive Gaussian", "Adaptive Mean", "Otsu (automatic)", "Global", "Grayscale (no threshold)"]
DEFAULTS = [METHODS[0], 0, 1.0, 170, 31, 10, 0.0, 0, 0.0, False]


def validate_file(path: str) -> Path:
    if not path or not Path(path).is_file():
        raise ValueError("Upload a PDF, PNG, or JPEG first.")
    file = Path(path)
    if file.stat().st_size > MAX_UPLOAD_BYTES:
        raise ValueError("The upload limit is 50 MB. Split or compress this file first.")
    if file.suffix.lower() not in {".pdf", ".png", ".jpg", ".jpeg"}:
        raise ValueError("Supported uploads: PDF, PNG, JPG, and JPEG.")
    return file


def inspect_document(path: str) -> int:
    file = validate_file(path)
    if file.suffix.lower() != ".pdf":
        with Image.open(file) as image:
            if image.width * image.height > MAX_PAGE_PIXELS:
                raise ValueError("This image exceeds 18 million pixels. Resize it before uploading.")
        return 1
    with PDF_LOCK, pymupdf.open(file) as doc:
        if doc.needs_pass:
            raise ValueError("Password-protected PDFs are not supported. Upload an unlocked copy.")
        if not 1 <= len(doc) <= MAX_PAGES:
            raise ValueError("Use a PDF with 1–100 pages. Split longer scores first.")
        return len(doc)


def render_page(path: str, page_number: int, dpi: int) -> dict:
    """Render one PDF page; do not resample native image uploads."""
    file = validate_file(path)
    dpi = int(dpi)
    if dpi not in {150, 200, 300}:
        raise ValueError("Choose 150, 200, or 300 DPI.")
    page_number = int(page_number)
    if file.suffix.lower() == ".pdf":
        with PDF_LOCK, pymupdf.open(file) as doc:
            if doc.needs_pass:
                raise ValueError("Password-protected PDFs are not supported.")
            if not 1 <= len(doc) <= MAX_PAGES or not 1 <= page_number <= len(doc):
                raise ValueError("Page number is outside the supported document range.")
            page = doc[page_number - 1]
            size_points = (float(page.rect.width), float(page.rect.height))
            expected = math.ceil(size_points[0] * dpi / 72) * math.ceil(size_points[1] * dpi / 72)
            if expected > MAX_PAGE_PIXELS:
                raise ValueError("This page would exceed 18 million pixels. Select a lower DPI.")
            pix = page.get_pixmap(dpi=dpi, colorspace=pymupdf.csGRAY, alpha=False)
            gray = np.frombuffer(pix.samples, np.uint8).reshape(pix.height, pix.stride)[:, :pix.width].copy()
            count = len(doc)
        resolution_note = f"PDF rendered at {dpi} DPI."
        output_dpi = (dpi, dpi)
    else:
        if page_number != 1:
            raise ValueError("Image uploads have only one page.")
        with Image.open(file) as source:
            if source.width * source.height > MAX_PAGE_PIXELS:
                raise ValueError("This image exceeds 18 million pixels.")
            source_dpi = source.info.get("dpi", (dpi, dpi))
            image = ImageOps.exif_transpose(source).convert("RGBA")
            background = Image.new("RGBA", image.size, "white")
            background.alpha_composite(image)
            gray = np.array(background.convert("L"))
        try:
            dx, dy = float(source_dpi[0]), float(source_dpi[1])
            if not (10 <= dx <= 2400 and 10 <= dy <= 2400):
                raise ValueError
        except (TypeError, ValueError, IndexError):
            dx = dy = dpi
        # Native pixels are unchanged; DPI only determines physical print dimensions.
        output_dpi = (dx, dy)
        size_points = (gray.shape[1] * 72 / dx, gray.shape[0] * 72 / dy)
        count = 1
        resolution_note = "Image kept at native pixels; DPI is print metadata only."
    return {"gray": gray, "size_points": size_points, "output_dpi": output_dpi,
            "path": str(file), "page": page_number, "dpi": dpi, "count": count,
            "resolution_note": resolution_note}




Writing app.py


## 3. Image arithmetic and thresholding

**Concept group 1:** `cv2.add` and `cv2.subtract` adjust brightness without uint8 overflow. `cv2.multiply` scales contrast around mid-gray, and `np.clip` prevents wrapping when converting floats back to uint8.

**Concept group 2:** global threshold uses one cutoff; adaptive mean/Gaussian use a local neighborhood for uneven illumination; Otsu chooses a global cutoff automatically. `cv2.bitwise_not` supplies inverse polarity. Odd adaptive neighborhoods are enforced. Continuous grayscale is an alternative when faint notation is lost.

Shadow normalization, median denoising, and sharpening are optional. Denoising and sharpening are off by default to protect small musical marks. The function returns both the displayed result and a normal-polarity copy for alpha masks and diagnostics.

In [4]:
%%writefile -a app.py
def clean_page(gray: np.ndarray, method: str = DEFAULTS[0], brightness: int = 0,
               contrast: float = 1.0, threshold: int = 170, block_size: int = 31,
               offset: int = 10, shadow_strength: float = 0.0, denoise: int = 0,
               sharpen: float = 0.0, invert: bool = False) -> tuple[np.ndarray, np.ndarray, str]:
    """Return the displayed result and a canonical black-on-white copy.

    All arithmetic is saturating or explicitly clipped. Filtering is optional
    because thin staff lines and staccato dots are meaningful musical content.
    """
    if method not in METHODS:
        raise ValueError("Select a supported threshold method.")
    work = gray.copy()
    if shadow_strength > 0:
        # A broad dilation estimates paper brightness; division reduces slow lighting variation.
        kernel = np.ones((31, 31), np.uint8)
        paper = cv2.dilate(work, kernel)
        paper = cv2.GaussianBlur(paper, (0, 0), 15)
        normalized = cv2.divide(work, np.maximum(paper, 1), scale=255)
        work = cv2.addWeighted(work, 1 - float(shadow_strength), normalized, float(shadow_strength), 0)
    shift = np.full_like(work, min(100, abs(int(brightness))))
    work = cv2.add(work, shift) if brightness >= 0 else cv2.subtract(work, shift)
    # Scale contrast around middle gray, not around zero. Never wrap uint8 values.
    floating = work.astype(np.float32) - 128.0
    gain = np.full_like(floating, float(contrast))
    work = np.clip(cv2.multiply(floating, gain) + 128.0, 0, 255).astype(np.uint8)
    denoise = int(denoise)
    if denoise in {3, 5}:
        work = cv2.medianBlur(work, denoise)
    if sharpen > 0:
        blur = cv2.GaussianBlur(work, (0, 0), 1.0)
        work = cv2.addWeighted(work, 1 + float(sharpen), blur, -float(sharpen), 0)
    detail = ""
    if method.startswith("Adaptive"):
        # The OpenCV neighborhood must be odd and greater than one.
        block_size = max(3, min(151, int(block_size)))
        block_size += (block_size % 2 == 0)
        adaptive = cv2.ADAPTIVE_THRESH_GAUSSIAN_C if method == METHODS[0] else cv2.ADAPTIVE_THRESH_MEAN_C
        canonical = cv2.adaptiveThreshold(work, 255, adaptive, cv2.THRESH_BINARY, block_size, float(offset))
        detail = f"Neighborhood {block_size}px; offset {offset}."
    elif method == "Otsu (automatic)":
        chosen, canonical = cv2.threshold(work, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
        detail = f"Otsu chose threshold {chosen:.0f}."
    elif method == "Global":
        _, canonical = cv2.threshold(work, int(threshold), 255, cv2.THRESH_BINARY)
        detail = f"Global threshold {threshold}."
    else:
        canonical = work
        detail = "Continuous grayscale retained."
    final = cv2.bitwise_not(canonical) if invert else canonical.copy()
    return final, canonical, detail




Appending to app.py


## 4. Optional morphology and drawing diagnostics

A horizontal morphological opening estimates long line segments. A row projection groups candidate line rows; connected components find connected ink regions. The analytical overlay draws rectangles using `cv2.rectangle`. None of these heuristics identify musical pitches.

**Staff subtraction happens only inside this diagnostic function.** It does not modify the processed reading copy. Boxes and red line markings are never included in downloads.

In [5]:
%%writefile -a app.py
def diagnostics(canonical: np.ndarray) -> tuple[np.ndarray, np.ndarray, str]:
    """Reproducible analysis overlays; NOT optical music recognition or an export filter."""
    _, ink = cv2.threshold(canonical, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    h, w = canonical.shape
    staff_mask = cv2.morphologyEx(ink, cv2.MORPH_OPEN,
                                cv2.getStructuringElement(cv2.MORPH_RECT, (max(15, w // 25), 1)))
    # Staff subtraction is for component analysis only; the score export is untouched.
    symbols = cv2.subtract(ink, staff_mask)
    n, _, stats, _ = cv2.connectedComponentsWithStats(symbols, connectivity=8)
    overlay = cv2.cvtColor(canonical, cv2.COLOR_GRAY2RGB)
    overlay[staff_mask > 0] = (180, 45, 55)
    shown = 0
    for x, y, bw, bh, area in stats[1:]:
        if 15 <= area <= 0.02 * h * w and shown < 200:
            cv2.rectangle(overlay, (int(x), int(y)), (int(x + bw - 1), int(y + bh - 1)),
                          (40, 125, 95), 1)
            shown += 1
    profile = np.count_nonzero(staff_mask, axis=1)
    rows = np.where(profile > 0.5 * profile.max())[0] if profile.max() else np.array([], dtype=int)
    groups = 0
    last = -10
    for row in rows:
        if row - last > 2:
            groups += 1
        last = row
    note = (f"Heuristic: {groups} horizontal line groups; {shown} component boxes shown "
            f"(of {n - 1} components). These are not recognized notes or guaranteed staves.")
    return overlay, cv2.bitwise_not(staff_mask), note




Appending to app.py


## 5. Alpha masks and faithful export

**Concept group 3:** `cv2.bitwise_not` turns white paper / black ink into an alpha mask; `cv2.merge` adds that mask as a fourth channel. Optional transparent PNG uses black ink, or white ink if inversion is selected. JPEG and PDF remain opaque.

Current-page exports produce direct files. All-page image exports produce ZIP archives; PDF contains every cleaned page in order. The current settings are applied uniformly to all pages. The PDF uses lossless raster images and retains page dimensions, but it does not retain editable notation, original vectors, links, or searchable text.

Export processes one page at a time and limits total pixels to avoid runaway memory use. Each export gets unique paths; original uploads are never overwritten.

In [6]:
%%writefile -a app.py
def make_png(final: np.ndarray, canonical: np.ndarray, transparent: bool,
             invert: bool, dpi: tuple[float, float]) -> bytes:
    if transparent:
        # Alpha = foreground ink strength. Paper becomes transparent; antialiasing is retained.
        alpha = cv2.bitwise_not(canonical)
        channel = np.full_like(alpha, 255 if invert else 0)
        image = Image.fromarray(cv2.merge([channel, channel, channel, alpha]))
    else:
        image = Image.fromarray(final)
    stream = io.BytesIO()
    image.save(stream, format="PNG", dpi=dpi)
    return stream.getvalue()


def make_jpeg(final: np.ndarray, dpi: tuple[float, float]) -> bytes:
    stream = io.BytesIO()
    Image.fromarray(final).convert("RGB").save(stream, format="JPEG", quality=95, subsampling=0, dpi=dpi)
    return stream.getvalue()


def export_document(path: str, page_number: int, dpi: int, settings: list,
                    scope: str, transparent: bool, output_dir: str, progress=None) -> tuple[str, str, str, str]:
    """Export at the same resolution/settings as preview, one page at a time."""
    count = inspect_document(path)
    if scope not in {"Current page", "All pages"}:
        raise ValueError("Choose current-page or all-pages export.")
    pages = list(range(1, count + 1)) if scope == "All pages" else [int(page_number)]
    # Preflight total work to avoid oversized all-page exports.
    if Path(path).suffix.lower() == ".pdf":
        with PDF_LOCK, pymupdf.open(path) as source:
            total_pixels = 0
            for number in pages:
                if not 1 <= number <= len(source):
                    raise ValueError("Select a page in this document.")
                rect = source[number - 1].rect
                pixels = math.ceil(rect.width * int(dpi) / 72) * math.ceil(rect.height * int(dpi) / 72)
                if pixels > MAX_PAGE_PIXELS:
                    raise ValueError("A page exceeds 18 million pixels. Select a lower DPI.")
                total_pixels += pixels
            if total_pixels > MAX_EXPORT_PIXELS:
                raise ValueError("This export exceeds 240 million pixels. Lower DPI or split the PDF.")
    folder = Path(output_dir) / uuid.uuid4().hex
    folder.mkdir(parents=True, exist_ok=True)
    multi = len(pages) > 1
    png_path = folder / ("cleaned-png-pages.zip" if multi else "cleaned.png")
    jpg_path = folder / ("cleaned-jpeg-pages.zip" if multi else "cleaned.jpg")
    pdf_path = folder / "cleaned.pdf"
    png_zip = zipfile.ZipFile(png_path, "w", zipfile.ZIP_DEFLATED) if multi else None
    jpg_zip = zipfile.ZipFile(jpg_path, "w", zipfile.ZIP_STORED) if multi else None
    result_pdf = pymupdf.open()
    try:
        for index, number in enumerate(pages):
            if progress is not None:
                progress(index / len(pages), desc=f"Cleaning page {number}")
            rendered = render_page(path, number, int(dpi))
            final, canonical, _ = clean_page(rendered["gray"], *settings)
            inverted = bool(settings[-1])
            png_bytes = make_png(final, canonical, transparent, inverted, rendered["output_dpi"])
            jpeg_bytes = make_jpeg(final, rendered["output_dpi"])
            if multi:
                png_zip.writestr(f"page-{number:03d}.png", png_bytes)
                jpg_zip.writestr(f"page-{number:03d}.jpg", jpeg_bytes)
            else:
                png_path.write_bytes(png_bytes)
                jpg_path.write_bytes(jpeg_bytes)
            # Use an opaque lossless PNG inside the PDF; keep exact physical page dimensions.
            opaque_png = make_png(final, canonical, False, inverted, rendered["output_dpi"])
            with PDF_LOCK:
                out_page = result_pdf.new_page(width=rendered["size_points"][0],
                                              height=rendered["size_points"][1])
                out_page.insert_image(out_page.rect, stream=opaque_png)
        with PDF_LOCK:
            result_pdf.save(pdf_path, garbage=4, deflate=True)
    except Exception:
        if png_zip:
            png_zip.close()
            jpg_zip.close()
        shutil.rmtree(folder, ignore_errors=True)
        raise
    finally:
        with PDF_LOCK:
            result_pdf.close()
        if png_zip:
            png_zip.close()
            jpg_zip.close()
    note = (f"Exported {len(pages)} page(s) using the current controls. "
            "PNG is lossless; JPEG is lossy; PDF contains opaque raster pages, not editable notation. "
            + ("Image downloads are ZIP files. " if multi else "")
            + ("Transparency applies only to PNG." if transparent else ""))
    return str(png_path), str(jpg_path), str(pdf_path), note




Appending to app.py


## 6. Gradio callbacks and session state

A session-specific cache avoids rerendering the PDF when only a cleaning control changes. New settings invalidate existing download links. The callbacks report errors in the interface and keep exports separate from diagnostic overlays.

In [7]:
%%writefile -a app.py
def upload_choices(path):
    if not path:
        return gr.update(choices=[("Page 1", 1)], value=1)
    try:
        count = inspect_document(path)
        return gr.update(choices=[(f"Page {i}", i) for i in range(1, count + 1)], value=1)
    except Exception as exc:
        raise gr.Error(str(exc)) from exc


def preview(path, page_number, dpi, *args):
    settings = list(args[:-2])
    cache, show_diagnostics = args[-2:]
    if not path:
        return None, None, None, None, "Upload a PDF or image to begin.", None, None, None, None, ""
    try:
        key = (str(path), int(page_number or 1), int(dpi))
        if cache is None or (cache["path"], cache["page"], cache["dpi"]) != key:
            cache = render_page(*key)
        final, canonical, detail = clean_page(cache["gray"], *settings)
        overlay = staff = None
        diagnostic_note = ""
        if show_diagnostics:
            overlay, staff, diagnostic_note = diagnostics(canonical)
        h, w = final.shape
        note = (f"Page {cache['page']} of {cache['count']} | {w} × {h} pixels | "
                f"{cache['resolution_note']} {detail}\n\n{diagnostic_note}")
        # Any preview change invalidates old downloads so they cannot silently be stale.
        return cache["gray"], final, overlay, staff, note, cache, None, None, None, ""
    except Exception as exc:
        raise gr.Error(str(exc)) from exc


def export_ui(path, page_number, dpi, *args, progress=EXPORT_PROGRESS):
    settings = list(args[:-3])
    scope, transparent, output_dir = args[-3:]
    try:
        return export_document(path, page_number, int(dpi), settings, scope, transparent,
                               output_dir, progress=progress)
    except Exception as exc:
        raise gr.Error(str(exc)) from exc


def cleanup_session(path):
    shutil.rmtree(path, ignore_errors=True)




Appending to app.py


## 7. Build the interface

The narrow controls panel sits beside original/enhanced previews. Gradio supplies native upload, fullscreen, and download controls. Slider changes refresh the preview on release; dropdowns and checkboxes refresh immediately.

Try Adaptive Gaussian, neighborhood 31, offset 10, brightness 0, contrast 1.0 first. Zoom into notes, accidentals, and dots before exporting. The file limits are 50 MB, 100 PDF pages, 18 million pixels per page, and 240 million pixels per export.

In [8]:
%%writefile -a app.py
def build_app() -> gr.Blocks:
    with gr.Blocks(title="Document Cleaner", delete_cache=(600, 3600)) as demo:
        cache = gr.State(value=None, time_to_live=3600)
        session_dir = gr.State(value=lambda: tempfile.mkdtemp(prefix="score-cleaner-"),
                               time_to_live=3600, delete_callback=cleanup_session)
        gr.Markdown("# Document Cleaner\nClean uneven lighting. Keep the music intact.\n\n"
                    "Upload a score, compare the original with the result, then export. "
                    "No generative reconstruction or music recognition is used.")
        with gr.Row():
            with gr.Column(scale=1, min_width=290):
                file = gr.File(label="Upload PDF or image", file_types=[".pdf", ".png", ".jpg", ".jpeg"],
                               type="filepath")
                with gr.Row():
                    page = gr.Dropdown(choices=[("Page 1", 1)], value=1, label="Page")
                    dpi = gr.Dropdown(choices=[150, 200, 300], value=200, label="PDF render DPI")
                method = gr.Dropdown(METHODS, value=METHODS[0], label="Cleaning method")
                brightness = gr.Slider(-100, 100, value=0, step=1, label="Brightness")
                contrast = gr.Slider(0.5, 2.0, value=1, step=0.05, label="Contrast")
                threshold = gr.Slider(0, 255, value=170, step=1, label="Global threshold (Global only)")
                block = gr.Slider(3, 151, value=31, step=2, label="Adaptive neighborhood (adaptive only)")
                offset = gr.Slider(-20, 30, value=10, step=1, label="Adaptive offset (adaptive only)")
                with gr.Accordion("Advanced adjustments", open=False):
                    shadows = gr.Slider(0, 1, value=0, step=0.1, label="Shadow normalization")
                    denoise = gr.Dropdown([("Off — preserve thin detail", 0), ("Median 3 × 3", 3),
                                          ("Median 5 × 5", 5)], value=0, label="Denoise")
                    sharp = gr.Slider(0, 1, value=0, step=0.1, label="Sharpen")
                    inverse = gr.Checkbox(False, label="Invert black / white")
                    gr.Markdown("Filtering is off by default. Median filters and strong thresholds "
                                "can erase dots, accidentals, and thin lines. Check details at full size.")
                reset = gr.Button("Reset cleaning controls")
            with gr.Column(scale=3):
                with gr.Row():
                    original = gr.Image(label="Original · rendered grayscale", interactive=False,
                                        type="numpy", height=530)
                    enhanced = gr.Image(label="Enhanced · export preview", interactive=False,
                                        type="numpy", height=530)
                status = gr.Markdown("Upload a PDF or image to begin.")
                with gr.Accordion("Notebook diagnostics · never included in downloads", open=False):
                    show_diagnostics = gr.Checkbox(False, label="Show staff-line and component analysis")
                    with gr.Row():
                        overlay = gr.Image(label="Line mask (red), component boxes (green)",
                                           interactive=False, height=350)
                        staff = gr.Image(label="Horizontal-line mask only", interactive=False, height=350)
                    gr.Markdown("These are experimental OpenCV heuristics, not detected pitches. "
                                "Staff lines are removed only for this analysis, never from the exported score.")
        gr.Markdown("## Export\nControls update the preview after a slider is released. "
                    "Changing settings clears old downloads. PNG is recommended for music detail.")
        with gr.Row():
            scope = gr.Radio(["Current page", "All pages"], value="Current page", label="Export pages")
            transparent = gr.Checkbox(False, label="Transparent paper in PNG only")
            export_button = gr.Button("Generate downloads", variant="primary")
        with gr.Row():
            png = gr.DownloadButton(label="Download PNG / PNG ZIP", value=None)
            jpg = gr.DownloadButton(label="Download JPEG / JPEG ZIP", value=None)
            pdf = gr.DownloadButton(label="Download PDF", value=None)
        export_status = gr.Markdown()
        gr.Markdown("Limits: 50 MB upload, 100 pages, 18 million pixels per page, "
                    "240 million pixels per export. PDF output is image-only: original searchable text, "
                    "vectors, links, and annotations are not preserved as editable objects. "
                    "Uploaded files are processed on the machine hosting this app, not by an AI API.")

        controls = [method, brightness, contrast, threshold, block, offset, shadows, denoise, sharp, inverse]
        inputs = [file, page, dpi, *controls, cache, show_diagnostics]
        outputs = [original, enhanced, overlay, staff, status, cache, png, jpg, pdf, export_status]
        event_options = {"concurrency_id": "document_work", "concurrency_limit": 1,
                         "trigger_mode": "always_last"}
        file.change(upload_choices, inputs=file, outputs=page, **event_options).then(
            preview, inputs=inputs, outputs=outputs, **event_options)
        for control in [page, dpi, method, denoise, inverse, show_diagnostics]:
            control.change(preview, inputs=inputs, outputs=outputs, **event_options)
        for slider in [brightness, contrast, threshold, block, offset, shadows, sharp]:
            slider.release(preview, inputs=inputs, outputs=outputs, **event_options)
        reset.click(lambda: DEFAULTS.copy(), outputs=controls, **event_options).then(
            preview, inputs=inputs, outputs=outputs, **event_options)
        for export_control in [scope, transparent]:
            export_control.change(lambda: (None, None, None, ""), outputs=[png, jpg, pdf, export_status],
                                  **event_options)
        export_button.click(export_ui, inputs=[file, page, dpi, *controls, scope, transparent, session_dir],
                            outputs=[png, jpg, pdf, export_status], **event_options)
    return demo.queue(default_concurrency_limit=1, max_size=16)


if __name__ == "__main__":
    build_app().launch(server_name="0.0.0.0", server_port=int(os.environ.get("PORT", "7860")),
                       share=False, max_file_size="50mb")


Appending to app.py


## 8. A tiny pixel-array sanity check

This small test demonstrates saturation rather than overflow and checks that inverse polarity does not change the canonical reading copy. The full package contains more complete tests for PDF page order, dimensions, PNG alpha, exports, and input limits.

In [9]:
import importlib
import app
import numpy as np

importlib.reload(app)
pixels = np.array([[0, 100, 200, 250]], dtype=np.uint8)
bright, _, _ = app.clean_page(pixels, method='Grayscale (no threshold)', brightness=30)
assert bright.tolist() == [[30, 130, 230, 255]]
normal, canonical, _ = app.clean_page(pixels, method='Global', threshold=170)
inverse, inverse_canonical, _ = app.clean_page(pixels, method='Global', threshold=170, invert=True)
assert np.array_equal(inverse, 255 - normal)
assert np.array_equal(canonical, inverse_canonical)
print('Original pixels:', pixels.tolist())
print('Brightness +30:', bright.tolist())
print('Pixel-array checks passed.')


Original pixels: [[0, 100, 200, 250]]
Brightness +30: [[30, 130, 230, 255]]
Pixel-array checks passed.


## 9. Write the Hugging Face configuration

Spaces needs `app.py`, `requirements.txt`, and `README.md` at the repository root. The README front matter selects the Gradio SDK and the tested runtime version. You do not deploy a live Jupyter kernel. Review library licensing and choose a Space license yourself; PyMuPDF uses AGPL-3.0 or a separate commercial license.

Create a Space at https://huggingface.co/new-space, select Gradio and CPU, then upload those three files. Select Public only if you want public source and a no-login app. Do not commit private scores or secrets. This notebook does not publish anything automatically.

In [10]:
%%writefile README.md
---
title: Document Cleaner
colorFrom: gray
colorTo: green
sdk: gradio
sdk_version: 6.30.0
python_version: "3.11"
app_file: app.py
pinned: false
---

# Document Cleaner

A Gradio app for cleaning scanned or photographed sheet music in PDFs. It also accepts PNG and JPEG uploads. Processing uses **OpenCV and PyMuPDF**, not an AI model. No API key or GPU is needed.

## Start in Jupyter

Open `Document_Cleaner.ipynb` in a fresh Python 3.11+ Jupyter kernel and run the cells in order. The notebook is self-contained: its cells write `requirements.txt`, `app.py`, and this Space configuration before launching the interface inline. Run it in a dedicated writable working folder; the write-file cells overwrite those three files, so back up later edits before rerunning them. If you have imported different NumPy/OpenCV versions into the kernel already, restart the kernel after installation, then rerun the cells.

If you downloaded the full package, you can also install and launch from a terminal:

```bash
python -m venv .venv
# macOS / Linux:
source .venv/bin/activate
# Windows PowerShell instead:
# .venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
python app.py
```

Open the local address printed by Gradio. Jupyter itself is a development dependency, not a runtime dependency for Spaces; install `jupyterlab` separately if you do not already have it.

## Use the cleaner

Upload a PDF, PNG, or JPEG. Select a page and choose 150, 200, or 300 DPI for PDFs. Begin with **Adaptive Gaussian**, brightness 0, contrast 1.0, neighborhood 31, and offset 10. Preview changes occur when a slider is released or a dropdown/checkbox is changed. Use Gradio's image fullscreen controls to inspect small notes and accidentals before exporting.

Global threshold uses the Global threshold slider. Adaptive Mean and Adaptive Gaussian use the neighborhood and offset sliders. Otsu chooses its threshold automatically. Grayscale retains continuous tones. The sliders that do not apply to the selected method have no effect on that method.

Advanced shadow normalization, median denoising, and sharpening are optional. These filters are off by default because strong adjustments can remove thin staff lines or staccato dots. Reset restores the cleaning controls but leaves the uploaded file, page, and DPI unchanged.

Click **Generate downloads** after choosing current-page or all-page export. Single-page image exports are direct PNG/JPEG files. Multi-page image exports are ZIP archives containing one image per page. PDF export is always a PDF and preserves page order and physical page dimensions. All-page export uses the **same current controls for every page**; settings are not saved separately for each page.

PNG is lossless and recommended. JPEG is lossy. Optional transparent paper applies only to PNG. PDF and JPEG are opaque; with normal polarity they have white paper, and inversion makes them white ink on black paper. Transparent PNG uses black ink normally or white ink when inverted.

For image uploads, native pixels are preserved. Embedded DPI is used when valid; otherwise the DPI control sets print metadata and the output PDF's physical size. It does not create extra image detail.

## Notebook concepts used

The image arithmetic mirrors the provided notebook: `cv2.add`, `cv2.subtract`, `cv2.multiply`, and `np.clip` safely adjust brightness and contrast. Thresholding includes global, inverse polarity, Otsu, adaptive mean, and adaptive Gaussian. Transparent PNG uses `cv2.bitwise_not` and `cv2.merge` to add an alpha mask. These are three distinct concept groups from the assignment.

The optional diagnostic view demonstrates horizontal morphology, projection-based line grouping, connected components, and `cv2.rectangle` drawing. It is a reproducible analytical overlay, not optical music recognition. Staff-line subtraction is used only to analyze connected components. **The exported music retains its staff lines and never includes diagnostic markings.**

YOLO, note naming, synthesized playback, and blueprint-style coloring are intentionally excluded: they do not improve document cleaning, and the notebook's note-naming example makes score-specific assumptions.

## Deploy on Hugging Face Spaces

1. Sign in to Hugging Face and open [Create a new Space](https://huggingface.co/new-space).
2. Give it a name such as `document-cleaner`, choose **Gradio**, and choose a license after reviewing the dependency licenses. Select **Public** if your assignment requires a link that opens without login. Public Spaces expose your source code; do not commit private sheet music or credentials.
3. Choose a CPU environment. This pipeline does not need a GPU. Resource availability and current hosting terms are controlled by Hugging Face.
4. Upload **`app.py`, `requirements.txt`, and `README.md`** into the repository root. Replace the auto-generated README with the supplied configuration rather than nesting these files in a folder. The notebook can be uploaded as educational source but is not the deployed entry point.
5. Wait for the build to finish, open the **App** tab, and test a score. Inspect build/runtime logs if it fails.
6. Open the Space link in a signed-out/private browser window to confirm no-login access before submitting it for your assignment.

Hugging Face runs the Python entry point, not a live Jupyter kernel. Developing in Jupyter and deploying the same functions in `app.py` gives you both a learning notebook and a normal hosted app. The temporary Gradio `share=True` link is not the same thing as a deployed Hugging Face Space.

## Limits and privacy

This starter limits uploads to 50 MB, PDFs to 100 pages, each rendered page to 18 million pixels, and each export to 240 million pixels. Split a large PDF or reduce render DPI when needed. Password-protected PDFs are rejected. Preview caches are session-specific; exports use unique session directories. Gradio cache and session cleanup are configured for approximately one-hour retention, but this is not a guarantee of immediate deletion or confidentiality on a shared host.

When deployed, uploads leave your computer and are processed on the hosting server. No document is sent to an AI inference API. Do not use a public demo for confidential, restricted, or unauthorized material. This is an educational starter, not a hardened production document service.

Cleaning cannot recover musical information that was not captured, such as an unreadable accidental or a blurred-away note. Review the result against the original. Generated PDFs contain raster images and do **not** retain editable notation, original vectors, searchable text, links, or annotations as editable PDF objects. Keep the original PDF as your master.

## Develop with Claude Code

Claude Code can help edit the notebook and the shared app, debug callbacks, add controls, and maintain tests. The app does not require Claude at runtime. See `CLAUDE_CODE_PROMPT.md` for a project-specific starting prompt and `GUIDE.md` for subscription-login guidance.

After changing `app.py`, run `python make_notebook.py` to regenerate the self-contained notebook from the same source. If you change notebook-generated app code first, run those write-file cells and then regenerate the notebook to keep both forms synchronized. Close the Gradio app before rerunning its launch cell.

Developer checks (optional):

```bash
python -m pip install pytest ruff nbformat nbclient ipykernel
ruff check app.py make_notebook.py tests
pytest -q
python make_notebook.py
```

## Open-source credits and licensing

PDF support is powered by [PyMuPDF](https://github.com/pymupdf/pymupdf), maintained by Artifex. PyMuPDF is offered under **AGPL-3.0** or a separate commercial license. Review its licensing before redistributing or turning this starter into a proprietary service. This package does not choose a license on your behalf.

The interface uses [Gradio](https://github.com/gradio-app/gradio), image operations use [OpenCV](https://github.com/opencv/opencv), and image encoding uses [Pillow](https://github.com/python-pillow/Pillow).

### Official references

- [Gradio Spaces](https://huggingface.co/docs/hub/en/spaces-sdks-gradio)
- [Space dependencies](https://huggingface.co/docs/hub/en/spaces-dependencies)
- [Gradio notebook development](https://gradio.app/guides/developing-faster-with-reload-mode)
- [PyMuPDF page rendering](https://pymupdf.readthedocs.io/en/latest/recipes-images.html)


Writing README.md


## 10. Launch inside Jupyter

Run this cell and use the interface below it. Releasing a slider updates the preview. Click Generate downloads to produce all three output formats.

For a local notebook, keep `share=False`. If your remote/Colab notebook cannot display a local server, you can intentionally change this to `share=True`; that creates a **temporary public Gradio link**, not a permanent Space. Do not use that link for confidential documents. On Hugging Face the app runs on the hosting server; no AI inference API receives your score.

In [11]:
# Close any old instance before rebuilding the interface.
if 'demo' in globals():
    demo.close()
importlib.reload(app)
demo = app.build_app()
if os.environ.get('DOCUMENT_CLEANER_TEST') == '1':
    print('Interface constructed successfully; automated test does not start a server.')
else:
    demo.launch(inline=True, share=False, max_file_size='50mb')


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

## 11. Continue with Claude Code

Claude Pro/Max subscribers can use subscription sign-in to develop this project with Claude Code. A developer subscription is not an inference API entitlement for your hosted app. No Claude API key is needed here. If `ANTHROPIC_API_KEY` is set in your environment, Anthropic says Claude Code may use API authentication/billing instead of subscription usage.

Official subscription guidance: https://support.claude.com/en/articles/11145838-use-claude-code-with-your-pro-or-max-plan

# Prompt for Claude Code

Open Claude Code in this project folder, then paste the following prompt. If you want it to consult the original course materials, first place `Build an App.odt` and `Copy_of_CV_Adding_Vectors.ipynb` in the folder too. The original files are not redistributed in this starter package.

> Help me develop this Document Cleaner for photographed or scanned sheet music. I want to learn through Jupyter Notebook, build the interface with Gradio, and deploy a standard Gradio Space on Hugging Face.
>
> First read README.md, GUIDE.md, app.py, Document_Cleaner.ipynb, and the existing tests. If the original course notebook and Build an App brief are present, consult them too. Explain the current structure in plain English before changing anything.
>
> Preserve these requirements: upload PDF or PNG/JPEG; navigate all PDF pages; choose render DPI; show the original and enhanced page; update the preview from sliders/dropdowns; export a selected page or all pages as PNG, JPEG, and PDF. Multiple PNG/JPEG pages should be ZIP archives. Keep original files unchanged and preserve page order and physical PDF page sizes.
>
> Demonstrate at least three course concept groups: safe OpenCV brightness/contrast arithmetic with clipping, global/inverse/Otsu/adaptive thresholding, and alpha masks for transparent PNG. Keep the staff-line morphology and connected-component drawing overlay in diagnostics only. Never remove staff lines from the exported score. Do not add YOLO, pitch recognition, or generative image reconstruction unless I explicitly request it.
>
> Avoid destructive default filters. Preserve tiny dots, stems, ledger lines, and accidentals. Explain that badly blurred or missing notation cannot be reliably recovered. No AI API or GPU should be necessary at runtime. Do not add Claude API calls merely because I am using Claude Code to develop the app.
>
> Reuse app.py as the deployment entry point and make_notebook.py to regenerate the self-contained notebook. Keep requirements.txt and the Gradio sdk_version in README.md consistent. Use functions with explanatory comments. Run the existing tests, and add focused tests for any changed behavior. Check that PNG matches the preview, transparent PNG alpha works, JPEG/PDF are opaque, and all-page export preserves order.
>
> Make the improvements I request, explain how I can run each notebook cell, and provide a short account of changes and test results. Do not publish a Space, upload my sheet music to another service, or commit credentials without my explicit authorization.

A useful first follow-up request is: “Walk me through the brightness, contrast, and adaptive-threshold functions one at a time, using tiny pixel arrays before we change any code.”


## Credits and official documentation

Your original course notebook supplies the concept sequence. PDF processing reuses [PyMuPDF](https://github.com/pymupdf/pymupdf), maintained by Artifex, rather than a custom PDF parser. Other libraries are Gradio, OpenCV, NumPy, and Pillow.

- [Gradio Spaces](https://huggingface.co/docs/hub/en/spaces-sdks-gradio)
- [Space dependencies](https://huggingface.co/docs/hub/en/spaces-dependencies)
- [Gradio notebook workflows](https://gradio.app/guides/developing-faster-with-reload-mode)
- [PyMuPDF rendering recipes](https://pymupdf.readthedocs.io/en/latest/recipes-images.html)
